# 13 · Reduce guides to their target genes

Up to this point a cell is labelled by the *guide* it received. From here it is
labelled by the *gene* that guide targets, which means pooling the cells that
received different guides against the same gene.

The screen carries several guides per gene, and each was delivered to a
different set of cells. Treated separately, each guide gives a weak estimate
from few cells. Pooled, they give one estimate per gene from all the cells
perturbed in it — which is what the effect models in notebooks 14 and 15 are
fitted on. So the guide-level columns of `.obs` are collapsed onto their
target: the several guides against *Cbl* become one `GENE_Cbl_` column, set for
any cell that received any of them.

**Which guides are pooled is decided by the selection in notebook 12.** Only
the guides that agreed with another guide against the same gene are kept; the
ones listed in `par_bad_KO_guides_file` are dropped before the pooling, and any
cell whose only guide was rejected drops out with them. That is why the cell
count falls here as well as the column count: pooling a guide that did not work
would dilute the very effect the gene-level estimate is meant to measure.

A cell is also assigned a target only if the guides it carries agree on one. In
the multiple-knockout population, cells left with a single distinct target after
pooling — two guides against the same gene rather than two different genes —
are not combinatorial perturbations and are dropped.

**Reads** `par_save_filename_7` and `par_save_filename_6`, and
`par_bad_KO_guides_file`.
**Writes** `par_save_filename_8` and `par_save_filename_9`.

Both objects get a `ClusterResiduals` layer: expression with the detected-gene
count, mitochondrial fraction and Leiden cluster regressed out. The effect
models in notebooks 14 and 15 are fitted on that layer rather than on `.X`.

## Setup

In [1]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)
from sklearn import linear_model

CONTROL_PREFIXES = (par_not_target_control_prefix, par_nongene_site_control_prefix)

## Helpers

A guide is named `<target>_<n>`, so the target is everything before the final
underscore. Control guides are excluded from the gene columns and counted in a
single `GENE_CONTROL_` column.

In [2]:
def target_of(guide):
    """Guide name to target gene: 'Cbl_2' -> 'Cbl_'."""
    return "_".join(guide.split("_")[:-1]) + "_"


def is_control(guide):
    return guide.startswith(CONTROL_PREFIXES)


def aggregate_guides_to_genes(adata, guide_names):
    """Collapse guide indicator columns in .obs onto their target gene."""
    carried = adata.obs[guide_names] > 0

    dominant = carried.idxmax(axis=1)
    adata.obs["Cell_category"] = [
        "CONTROL_" if is_control(g) else target_of(g) for g in dominant
    ]

    ko_guides = [g for g in guide_names if not is_control(g)]
    targets = pd.Series([target_of(g) for g in ko_guides], index=ko_guides)

    # Transpose-groupby-transpose: groupby(axis=1) was removed in pandas 2.
    per_gene = carried[ko_guides].T.groupby(targets).any().T
    per_gene.columns = "GENE_" + per_gene.columns

    adata.obs = adata.obs.join(per_gene.astype("int64"))
    adata.obs["GENE_CONTROL_"] = (adata.obs["Cell_category"] == "CONTROL_").astype("int64")

    adata.uns["feature_barcode_names_filtered"] = list(guide_names)
    adata.uns["feature_barcode_names_filtered_GENES"] = [
        c for c in adata.obs.columns if c.startswith("GENE_")
    ]
    adata.obs = adata.obs.drop(columns=list(guide_names)).copy()
    return adata


def add_cluster_residuals(adata):
    """Regress expression on n_genes, mt_frac and leiden; store the residuals."""
    rna = pd.DataFrame(
        adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X,
        index=adata.obs_names, columns=adata.var_names,
    )
    covars = adata.obs[["n_genes", "mt_frac", "leiden"]]
    covars = covars.join(pd.get_dummies(covars.leiden)).drop(columns=["leiden"])

    regr = linear_model.LinearRegression(fit_intercept=False).fit(covars, rna)
    adata.layers["ClusterResiduals"] = (rna - regr.predict(covars)).to_numpy()
    return adata

## Single knockouts

The rare-guide test runs again because the earlier filtering removed cells.

In [3]:
adata_single = sc.read(par_save_filename_7)
print(f"input: {adata_single.shape[0]} cells x {adata_single.shape[1]} genes")

guides = list(adata_single.uns["feature_barcode_names_filtered"])
carried = adata_single.obs[guides] > 0

bad_guides = set(pd.read_csv(par_bad_KO_guides_file)["x"])
drop = (carried.sum(axis=0) < par_ncell_test_threshold) | carried.columns.isin(bad_guides)
guides = [g for g in guides if not drop[g]]
print(f"guides kept after dropping rejected and rare guides: {len(guides)}")

adata_single = adata_single[(adata_single.obs[guides] > 0).sum(axis=1) > 0].copy()
print(f"cells still carrying a guide: {adata_single.shape[0]}")

adata_single = aggregate_guides_to_genes(adata_single, guides)
adata_single = add_cluster_residuals(adata_single)

gene_cols = adata_single.uns["feature_barcode_names_filtered_GENES"]
print(f"target genes (incl. CONTROL): {len(gene_cols)}")

adata_single.write(par_save_filename_8)
print(f"written: {par_save_filename_8}  ({adata_single.shape[0]} x {adata_single.shape[1]})")

input: 325203 cells x 6685 genes
guides kept after dropping rejected and rare guides: 2561
cells still carrying a guide: 246050
target genes (incl. CONTROL): 1032
written: outputs/anndata/adata-SingleKO_PerGENE.h5ad  (246050 x 6685)


## Multiple knockouts

This branch inherits the single-knockout object's gene set and surviving guide
list, so the two can be concatenated in notebook 17.

`GENE_INEFFECT_` marks cells left with only one distinct target after
aggregation — two guides against the same gene. They are not combinatorial
perturbations and are dropped.

In [4]:
adata_multiple = sc.read(par_save_filename_6)
adata_multiple = adata_multiple[:, adata_single.var_names].copy()
sc.pp.filter_cells(adata_multiple, min_genes=par_mincellgenes_for_testedgenes)
print(f"after cell filtering: {adata_multiple.shape[0]} cells")

adata_multiple = adata_multiple[
    (adata_multiple.obs[guides] > 0).sum(axis=1) > 0
].copy()
print(f"cells carrying a surviving guide: {adata_multiple.shape[0]}")

adata_multiple = aggregate_guides_to_genes(adata_multiple, guides)

gene_cols = adata_multiple.uns["feature_barcode_names_filtered_GENES"]
n_targets = adata_multiple.obs[gene_cols].sum(axis=1)
adata_multiple.obs["GENE_INEFFECT_"] = (n_targets == 1).astype("int64")
print(f"cells with only one distinct target (dropped): {int(adata_multiple.obs.GENE_INEFFECT_.sum())}")

adata_multiple = adata_multiple[adata_multiple.obs["GENE_INEFFECT_"] != 1].copy()
adata_multiple.uns["feature_barcode_names_filtered_GENES"] = [
    c for c in adata_multiple.obs.columns if c.startswith("GENE_")
]
adata_multiple = add_cluster_residuals(adata_multiple)

adata_multiple.write(par_save_filename_9)
print(f"written: {par_save_filename_9}  ({adata_multiple.shape[0]} x {adata_multiple.shape[1]})")

filtered out 6720 cells that have less than 800 genes expressed
after cell filtering: 171151 cells
cells carrying a surviving guide: 162085
cells with only one distinct target (dropped): 84212
written: outputs/anndata/adata-MultipleKO_PerGENE.h5ad  (77873 x 6685)
